<a href="https://colab.research.google.com/github/jamesemcnally/critical-listener/blob/main/recommender_1_model_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import random
from itertools import combinations
from sentence_transformers import SentenceTransformer

df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/merged_dataset_cleaned.parquet")
print(f"Loaded: {df.shape[0]:,} reviews")


Loaded: 48,189 reviews


## Comparing models:

1. MiniLM (512-token limit)
2. E5-large (512-token limit)
3. Nomic (8,192-token limit)

## Short-form album pairer prototype using MiniLM

In [5]:
import pandas as pd
import numpy as np
import random
from itertools import combinations
from sentence_transformers import SentenceTransformer

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

if 'df' not in dir():
    df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_cleaned.parquet")
    df["input_with_prefix"] = (
        "Artist: " + df["artist"].fillna("Unknown") + ". " +
        "Album: "  + df["album"].fillna("Unknown")  + ". " +
        df["cleaned_text"]
    )
    df["input_no_prefix"] = df["cleaned_text"]
    print(f"Loaded: {len(df):,} reviews")

if 'eval_pairs_sample' not in dir():
    random.seed(42)
    eval_pairs = []
    keyed = df.dropna(subset=["artist_norm", "album_norm"])
    for _, group in keyed.groupby(["artist_norm", "album_norm"]):
        if group["dataset"].nunique() > 1:
            for i, j in combinations(group.index.tolist(), 2):
                if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                    eval_pairs.append((i, j))
    eval_pairs_sample = random.sample(eval_pairs, 500)
    print(f"Eval pairs: {len(eval_pairs):,} | Sample: {len(eval_pairs_sample)}")

if 'evaluate_retrieval' not in dir():
    def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
        rr, hits = [], {k: [] for k in ks}
        for q_idx, t_idx in eval_pairs:
            sims = get_sims_fn(q_idx).copy()
            sims[q_idx] = -np.inf
            target_sim = sims[t_idx]
            same_album = df[
                (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
                (df["album_norm"]  == df.loc[q_idx, "album_norm"])
            ].index
            sims[same_album] = -np.inf
            sims[t_idx] = target_sim
            ranked = np.argsort(sims)[::-1]
            rank = int(np.where(ranked == t_idx)[0][0]) + 1
            rr.append(1.0 / rank)
            for k in ks:
                hits[k].append(int(rank <= k))
        return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}

if 'results' not in dir():
    results = {}

# MiniLM — load from Drive if already saved, otherwise encode
import os
for input_col, label, save_name in [
    ("input_with_prefix", "MiniLM (with prefix)", "minilm_with_prefix"),
    ("input_no_prefix",   "MiniLM (no prefix)",   "minilm_no_prefix"),
]:
    path = f"{DRIVE_DIR}/{save_name}.npy"
    if os.path.exists(path):
        embeddings = np.load(path)
        print(f"Loaded {save_name}.npy from Drive")
    else:
        if 'model_minilm' not in dir():
            model_minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        embeddings = model_minilm.encode(
            df[input_col].tolist(),
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        np.save(path, embeddings)
        print(f"Saved {save_name}.npy to Drive")

    if label not in results:
        def sims_fn(idx, embeddings=embeddings):
            return embeddings @ embeddings[idx]
        results[label] = evaluate_retrieval(sims_fn, eval_pairs_sample)

    print(f"\n{label}")
    print("-" * 40)
    for metric, val in results[label].items():
        print(f"  {metric}: {val:.4f}")

Eval pairs: 3,653 | Sample: 500
Loaded minilm_with_prefix.npy from Drive

MiniLM (with prefix)
----------------------------------------
  MRR: 0.6349
  Recall@1: 0.4920
  Recall@5: 0.8120
  Recall@10: 0.8940
Loaded minilm_no_prefix.npy from Drive

MiniLM (no prefix)
----------------------------------------
  MRR: 0.4176
  Recall@1: 0.2980
  Recall@5: 0.5580
  Recall@10: 0.6440


## Short-form album pairer prototype using E5-Large

In [6]:
import pandas as pd
import numpy as np
import random
import os
from itertools import combinations
from sentence_transformers import SentenceTransformer

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

if 'df' not in dir():
    df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_cleaned.parquet")
    df["input_with_prefix"] = (
        "Artist: " + df["artist"].fillna("Unknown") + ". " +
        "Album: "  + df["album"].fillna("Unknown")  + ". " +
        df["cleaned_text"]
    )
    df["input_no_prefix"] = df["cleaned_text"]
    print(f"Loaded: {len(df):,} reviews")

if 'eval_pairs_sample' not in dir():
    random.seed(42)
    eval_pairs = []
    keyed = df.dropna(subset=["artist_norm", "album_norm"])
    for _, group in keyed.groupby(["artist_norm", "album_norm"]):
        if group["dataset"].nunique() > 1:
            for i, j in combinations(group.index.tolist(), 2):
                if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                    eval_pairs.append((i, j))
    eval_pairs_sample = random.sample(eval_pairs, 500)
    print(f"Eval pairs: {len(eval_pairs):,} | Sample: {len(eval_pairs_sample)}")

if 'evaluate_retrieval' not in dir():
    def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
        rr, hits = [], {k: [] for k in ks}
        for q_idx, t_idx in eval_pairs:
            sims = get_sims_fn(q_idx).copy()
            sims[q_idx] = -np.inf
            target_sim = sims[t_idx]
            same_album = df[
                (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
                (df["album_norm"]  == df.loc[q_idx, "album_norm"])
            ].index
            sims[same_album] = -np.inf
            sims[t_idx] = target_sim
            ranked = np.argsort(sims)[::-1]
            rank = int(np.where(ranked == t_idx)[0][0]) + 1
            rr.append(1.0 / rank)
            for k in ks:
                hits[k].append(int(rank <= k))
        return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}

if 'results' not in dir():
    results = {}

# E5-large — load from Drive if already saved, otherwise encode
for input_col, label, save_name in [
    ("input_with_prefix", "E5-large (with prefix)", "e5_large_with_prefix"),
    ("input_no_prefix",   "E5-large (no prefix)",   "e5_large_no_prefix"),
]:
    path = f"{DRIVE_DIR}/{save_name}.npy"
    if os.path.exists(path):
        embeddings = np.load(path)
        print(f"Loaded {save_name}.npy from Drive")
    else:
        if 'model_e5' not in dir():
            model_e5 = SentenceTransformer("intfloat/e5-large-v2")
        inputs = ["passage: " + text for text in df[input_col].tolist()]
        embeddings = model_e5.encode(
            inputs,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        np.save(path, embeddings)
        print(f"Saved {save_name}.npy to Drive")

    if label not in results:
        def sims_fn(idx, embeddings=embeddings):
            return embeddings @ embeddings[idx]
        results[label] = evaluate_retrieval(sims_fn, eval_pairs_sample)

    print(f"\n{label}")
    print("-" * 40)
    for metric, val in results[label].items():
        print(f"  {metric}: {val:.4f}")

Loaded e5_large_with_prefix.npy from Drive

E5-large (with prefix)
----------------------------------------
  MRR: 0.9529
  Recall@1: 0.9220
  Recall@5: 0.9880
  Recall@10: 0.9960
Loaded e5_large_no_prefix.npy from Drive

E5-large (no prefix)
----------------------------------------
  MRR: 0.8453
  Recall@1: 0.7640
  Recall@5: 0.9460
  Recall@10: 0.9680


## Long-form album pairer prototype using Nomic

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

model_nomic = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True
)

for input_col, label, save_name in [
    ("input_with_prefix", "Nomic (with prefix)", "nomic_with_prefix_clean"),
    ("input_no_prefix",   "Nomic (no prefix)",   "nomic_no_prefix_clean"),
]:
    embeddings = model_nomic.encode(
        df[input_col].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    np.save(f"{DRIVE_DIR}/{save_name}.npy", embeddings)
    print(f"Saved {save_name}.npy to Drive")

    def sims_fn(idx, embeddings=embeddings):
        return embeddings @ embeddings[idx]

    result = evaluate_retrieval(sims_fn, eval_pairs_sample)
    print(f"\n{label}")
    print("-" * 40)
    for metric, val in result.items():
        print(f"  {metric}: {val:.4f}")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1506 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Saved nomic_with_prefix_clean.npy to Drive

Nomic (with prefix)
----------------------------------------
  MRR: 0.9403
  Recall@1: 0.9040
  Recall@5: 0.9860
  Recall@10: 0.9900


Batches:   0%|          | 0/1506 [00:00<?, ?it/s]

Saved nomic_no_prefix_clean.npy to Drive

Nomic (no prefix)
----------------------------------------
  MRR: 0.7832
  Recall@1: 0.6820
  Recall@5: 0.9080
  Recall@10: 0.9620


## Running with the masked dataset

## MiniLM

In [15]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import random
import os
from itertools import combinations

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_masked.parquet")
print(f"Loaded: {len(df):,} reviews")

df["input_masked_with_prefix"] = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text_masked"]
)
df["input_masked_no_prefix"] = df["cleaned_text_masked"]
print("Created masked input columns")

random.seed(42)
eval_pairs = []
keyed = df.dropna(subset=["artist_norm", "album_norm"])
for _, group in keyed.groupby(["artist_norm", "album_norm"]):
    if group["dataset"].nunique() > 1:
        for i, j in combinations(group.index.tolist(), 2):
            if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                eval_pairs.append((i, j))
eval_pairs_sample = random.sample(eval_pairs, 500)
print(f"Eval pairs: {len(eval_pairs):,} | Sample: {len(eval_pairs_sample)}")

def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
    rr, hits = [], {k: [] for k in ks}
    for q_idx, t_idx in eval_pairs:
        sims = get_sims_fn(q_idx).copy()
        sims[q_idx] = -np.inf
        target_sim = sims[t_idx]
        same_album = df[
            (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
            (df["album_norm"]  == df.loc[q_idx, "album_norm"])
        ].index
        sims[same_album] = -np.inf
        sims[t_idx] = target_sim
        ranked = np.argsort(sims)[::-1]
        rank = int(np.where(ranked == t_idx)[0][0]) + 1
        rr.append(1.0 / rank)
        for k in ks:
            hits[k].append(int(rank <= k))
    return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}

if 'masked_results' not in dir():
    masked_results = {}

for input_col, label, save_name in [
    ("input_masked_with_prefix", "MiniLM masked (with prefix)", "minilm_masked_with_prefix"),
    ("input_masked_no_prefix",   "MiniLM masked (no prefix)",   "minilm_masked_no_prefix"),
]:
    path = f"{DRIVE_DIR}/{save_name}.npy"
    if os.path.exists(path):
        embeddings = np.load(path)
        print(f"Loaded {save_name}.npy from Drive: {embeddings.shape}")
    else:
        print(f"{save_name}.npy not found — encoding now...")
        if 'model_minilm' not in dir():
            model_minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        embeddings = model_minilm.encode(
            df[input_col].tolist(),
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        np.save(path, embeddings)
        print(f"Saved {save_name}.npy to Drive")

    if label not in masked_results:
        def sims_fn(idx, embeddings=embeddings):
            return embeddings @ embeddings[idx]
        masked_results[label] = evaluate_retrieval(sims_fn, eval_pairs_sample)

    print(f"\n{label}")
    print("-" * 40)
    for metric, val in masked_results[label].items():
        print(f"  {metric}: {val:.4f}")

Loaded: 48,189 reviews
Created masked input columns
Eval pairs: 3,653 | Sample: 500
minilm_masked_with_prefix.npy not found — encoding now...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/753 [00:00<?, ?it/s]

Saved minilm_masked_with_prefix.npy to Drive

MiniLM masked (with prefix)
----------------------------------------
  MRR: 0.5506
  Recall@1: 0.4440
  Recall@5: 0.6740
  Recall@10: 0.7440
minilm_masked_no_prefix.npy not found — encoding now...


Batches:   0%|          | 0/753 [00:00<?, ?it/s]

Saved minilm_masked_no_prefix.npy to Drive

MiniLM masked (no prefix)
----------------------------------------
  MRR: 0.2462
  Recall@1: 0.1860
  Recall@5: 0.3140
  Recall@10: 0.3520


## E5-large (masked)

In [16]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import random
import os
from itertools import combinations

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_masked.parquet")
print(f"Loaded: {len(df):,} reviews")

df["input_masked_with_prefix"] = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text_masked"]
)
df["input_masked_no_prefix"] = df["cleaned_text_masked"]
print("Created masked input columns")

random.seed(42)
eval_pairs = []
keyed = df.dropna(subset=["artist_norm", "album_norm"])
for _, group in keyed.groupby(["artist_norm", "album_norm"]):
    if group["dataset"].nunique() > 1:
        for i, j in combinations(group.index.tolist(), 2):
            if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                eval_pairs.append((i, j))
eval_pairs_sample = random.sample(eval_pairs, 500)
print(f"Eval pairs: {len(eval_pairs):,} | Sample: {len(eval_pairs_sample)}")

def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
    rr, hits = [], {k: [] for k in ks}
    for q_idx, t_idx in eval_pairs:
        sims = get_sims_fn(q_idx).copy()
        sims[q_idx] = -np.inf
        target_sim = sims[t_idx]
        same_album = df[
            (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
            (df["album_norm"]  == df.loc[q_idx, "album_norm"])
        ].index
        sims[same_album] = -np.inf
        sims[t_idx] = target_sim
        ranked = np.argsort(sims)[::-1]
        rank = int(np.where(ranked == t_idx)[0][0]) + 1
        rr.append(1.0 / rank)
        for k in ks:
            hits[k].append(int(rank <= k))
    return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}

if 'masked_results' not in dir():
    masked_results = {}

for input_col, label, save_name in [
    ("input_masked_with_prefix", "E5-large masked (with prefix)", "e5_large_masked_with_prefix"),
    ("input_masked_no_prefix",   "E5-large masked (no prefix)",   "e5_large_masked_no_prefix"),
]:
    path = f"{DRIVE_DIR}/{save_name}.npy"
    if os.path.exists(path):
        embeddings = np.load(path)
        print(f"Loaded {save_name}.npy from Drive: {embeddings.shape}")
    else:
        print(f"{save_name}.npy not found — encoding now...")
        if 'model_e5' not in dir():
            model_e5 = SentenceTransformer("intfloat/e5-large-v2")
        inputs = ["passage: " + text for text in df[input_col].tolist()]
        embeddings = model_e5.encode(
            inputs,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        np.save(path, embeddings)
        print(f"Saved {save_name}.npy to Drive")

    if label not in masked_results:
        def sims_fn(idx, embeddings=embeddings):
            return embeddings @ embeddings[idx]
        masked_results[label] = evaluate_retrieval(sims_fn, eval_pairs_sample)

    print(f"\n{label}")
    print("-" * 40)
    for metric, val in masked_results[label].items():
        print(f"  {metric}: {val:.4f}")

Loaded: 48,189 reviews
Created masked input columns
Eval pairs: 3,653 | Sample: 500
e5_large_masked_with_prefix.npy not found — encoding now...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/1506 [00:00<?, ?it/s]

Saved e5_large_masked_with_prefix.npy to Drive

E5-large masked (with prefix)
----------------------------------------
  MRR: 0.9075
  Recall@1: 0.8500
  Recall@5: 0.9780
  Recall@10: 0.9900
e5_large_masked_no_prefix.npy not found — encoding now...


Batches:   0%|          | 0/1506 [00:00<?, ?it/s]

Saved e5_large_masked_no_prefix.npy to Drive

E5-large masked (no prefix)
----------------------------------------
  MRR: 0.7637
  Recall@1: 0.6900
  Recall@5: 0.8540
  Recall@10: 0.8920


## Nomic (masked)

In [18]:
import pandas as pd
import numpy as np
import random
import os
from itertools import combinations

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_masked.parquet")
print(f"Loaded: {len(df):,} reviews")

df["input_masked_with_prefix"] = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text_masked"]
)
df["input_masked_no_prefix"] = df["cleaned_text_masked"]
print("Created masked input columns")

random.seed(42)
eval_pairs = []
keyed = df.dropna(subset=["artist_norm", "album_norm"])
for _, group in keyed.groupby(["artist_norm", "album_norm"]):
    if group["dataset"].nunique() > 1:
        for i, j in combinations(group.index.tolist(), 2):
            if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                eval_pairs.append((i, j))
eval_pairs_sample = random.sample(eval_pairs, 500)
print(f"Eval pairs: {len(eval_pairs):,} | Sample: {len(eval_pairs_sample)}")

def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
    rr, hits = [], {k: [] for k in ks}
    for q_idx, t_idx in eval_pairs:
        sims = get_sims_fn(q_idx).copy()
        sims[q_idx] = -np.inf
        target_sim = sims[t_idx]
        same_album = df[
            (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
            (df["album_norm"]  == df.loc[q_idx, "album_norm"])
        ].index
        sims[same_album] = -np.inf
        sims[t_idx] = target_sim
        ranked = np.argsort(sims)[::-1]
        rank = int(np.where(ranked == t_idx)[0][0]) + 1
        rr.append(1.0 / rank)
        for k in ks:
            hits[k].append(int(rank <= k))
    return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}

if 'masked_results' not in dir():
    masked_results = {}

for path_name, label in [
    ("nomic_masked_with_prefix", "Nomic masked (with prefix)"),
    ("nomic_masked_no_prefix",   "Nomic masked (no prefix)"),
]:
    path = f"{DRIVE_DIR}/{path_name}.npy"
    if not os.path.exists(path):
        print(f"WARNING: {path_name}.npy not found — skipping")
        continue
    embeddings = np.load(path)
    print(f"Loaded {path_name}.npy: {embeddings.shape}")

    def sims_fn(idx, embeddings=embeddings):
        return embeddings @ embeddings[idx]

    masked_results[label] = evaluate_retrieval(sims_fn, eval_pairs_sample)
    print(f"\n{label}")
    print("-" * 40)
    for metric, val in masked_results[label].items():
        print(f"  {metric}: {val:.4f}")

Loaded: 48,189 reviews
Created masked input columns
Eval pairs: 3,653 | Sample: 500
Loaded nomic_masked_with_prefix.npy: (48189, 768)

Nomic masked (with prefix)
----------------------------------------
  MRR: 0.9335
  Recall@1: 0.8980
  Recall@5: 0.9820
  Recall@10: 0.9900
Loaded nomic_masked_no_prefix.npy: (48189, 768)

Nomic masked (no prefix)
----------------------------------------
  MRR: 0.6307
  Recall@1: 0.5540
  Recall@5: 0.7260
  Recall@10: 0.7680


## TF-IDF Baseline (with masked data)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

masked_inputs = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text_masked"].fillna("")
).tolist()

tfidf = TfidfVectorizer(max_features=100_000, min_df=2, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(masked_inputs)
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

def tfidf_sims(idx):
    sims = (tfidf_matrix @ tfidf_matrix[idx].T).toarray().flatten()
    return sims

results_tfidf_masked = evaluate_retrieval(tfidf_sims, eval_pairs_sample)
print("\nTF-IDF masked (with prefix)")
print("-" * 40)
for metric, val in results_tfidf_masked.items():
    print(f"  {metric}: {val:.4f}")

## Compare models

In [19]:
import numpy as np
import os

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

model_files = [
    ("minilm_masked_with_prefix",  "MiniLM masked (with prefix)"),
    ("minilm_masked_no_prefix",    "MiniLM masked (no prefix)"),
    ("e5_large_masked_with_prefix","E5-large masked (with prefix)"),
    ("e5_large_masked_no_prefix",  "E5-large masked (no prefix)"),
    ("nomic_masked_with_prefix",   "Nomic masked (with prefix)"),
    ("nomic_masked_no_prefix",     "Nomic masked (no prefix)"),
]

masked_results = {}
for save_name, label in model_files:
    path = f"{DRIVE_DIR}/{save_name}.npy"
    if not os.path.exists(path):
        print(f"WARNING: {save_name}.npy not found — skipping")
        continue
    embeddings = np.load(path)
    def sims_fn(idx, embeddings=embeddings):
        return embeddings @ embeddings[idx]
    masked_results[label] = evaluate_retrieval(sims_fn, eval_pairs_sample)

print("\nMASKED INPUT COMPARISON")
print(f"{'Model':<40} {'MRR':>8} {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("-" * 72)
print(f"{'TF-IDF masked (with prefix)':<40} "
      f"{results_tfidf_masked['MRR']:>8.4f} "
      f"{results_tfidf_masked['Recall@1']:>8.4f} "
      f"{results_tfidf_masked['Recall@5']:>8.4f} "
      f"{results_tfidf_masked['Recall@10']:>8.4f}")
for save_name, label in model_files:
    if label in masked_results:
        m = masked_results[label]
        print(f"{label:<40} {m['MRR']:>8.4f} {m['Recall@1']:>8.4f} "
              f"{m['Recall@5']:>8.4f} {m['Recall@10']:>8.4f}")
    else:
        print(f"{label:<40} {'(not found)':>8}")


MASKED INPUT COMPARISON
Model                                         MRR      R@1      R@5     R@10
------------------------------------------------------------------------
TF-IDF (with prefix)                       0.7942   0.6870   0.9380   0.9830
MiniLM masked (with prefix)                0.5506   0.4440   0.6740   0.7440
MiniLM masked (no prefix)                  0.2462   0.1860   0.3140   0.3520
E5-large masked (with prefix)              0.9075   0.8500   0.9780   0.9900
E5-large masked (no prefix)                0.7637   0.6900   0.8540   0.8920
BGE-M3 masked (with prefix)                0.9572   0.9300   0.9920   0.9940
BGE-M3 masked (no prefix)                  0.8142   0.7600   0.8800   0.9000
Nomic masked (with prefix)                 0.9335   0.8980   0.9820   0.9900
Nomic masked (no prefix)                   0.6307   0.5540   0.7260   0.7680


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import numpy as np
import scipy.sparse as sp

# Build TF-IDF on masked text with prefix
masked_inputs = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text_masked"].fillna("")
).tolist()

tfidf = TfidfVectorizer(max_features=100_000, min_df=2, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(masked_inputs)
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

def tfidf_sims(idx):
    vec = tfidf_matrix[idx]
    sims = (tfidf_matrix @ vec.T).toarray().flatten()
    return sims

results_tfidf_masked = evaluate_retrieval(tfidf_sims, eval_pairs_sample)
print("\nTF-IDF masked (with prefix)")
print("-" * 40)
for metric, val in results_tfidf_masked.items():
    print(f"  {metric}: {val:.4f}")

TF-IDF matrix: (48189, 100000)

TF-IDF masked (with prefix)
----------------------------------------
  MRR: 0.8806
  Recall@1: 0.8180
  Recall@5: 0.9660
  Recall@10: 0.9820


## Recommender prototype # 1



In [ ]:
import numpy as np
import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

# Reload unmasked parquet — masked df loaded by earlier cells lacks cleaned_text and input_with_prefix
df = pd.read_parquet(f"{DRIVE_DIR}/merged_dataset_cleaned.parquet")
df["input_with_prefix"] = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text"]
)
print(f"Loaded: {len(df):,} reviews")
print(df["input_with_prefix"].iloc[10][:200])

# Load clean Nomic embeddings
embeddings_with = np.load(f"{DRIVE_DIR}/nomic_with_prefix_clean.npy")
embeddings_no   = np.load(f"{DRIVE_DIR}/nomic_no_prefix_clean.npy")
embeddings = embeddings_with
print(f"Embeddings: {embeddings.shape}")

def recommend(query_album, query_artist=None, k=5, filter_same_artist=True):
    mask = df["album_norm"] == query_album.lower().strip()
    if query_artist:
        mask &= df["artist_norm"] == query_artist.lower().strip()
    query_reviews = df[mask]
    if len(query_reviews) == 0:
        print(f"Not found: '{query_album}'")
        return None
    query_artist_norm = df.loc[query_reviews.index[0], "artist_norm"]
    query_embedding = embeddings[query_reviews.index].mean(axis=0)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)
    sims = embeddings @ query_embedding
    sims[query_reviews.index] = -np.inf
    ranked = np.argsort(sims)[::-1]
    seen, results = set(), []
    for idx in ranked:
        artist_norm = df.loc[idx, "artist_norm"]
        if filter_same_artist and artist_norm == query_artist_norm:
            continue
        key = (artist_norm, df.loc[idx, "album_norm"])
        if key not in seen:
            seen.add(key)
            results.append({
                "album":   df.loc[idx, "album"],
                "artist":  df.loc[idx, "artist"],
                "source":  df.loc[idx, "dataset"],
                "score":   round(float(sims[idx]), 4),
                "snippet": df.loc[idx, "cleaned_text"][:300]
            })
        if len(results) >= k:
            break
    return results

recs = recommend("kid a", query_artist="radiohead")
for i, r in enumerate(recs, 1):
    print(f"\n{i}. {r['artist']} — {r['album']} ({r['source']}, score: {r['score']})")
    print(f"   {r['snippet'][:200]}...")

In [ ]:
# Recommender function

def recommend(query_album, query_artist=None, k=5, filter_same_artist=True):
    mask = df["album_norm"] == query_album.lower().strip()
    if query_artist:
        mask &= df["artist_norm"] == query_artist.lower().strip()

    query_reviews = df[mask]
    if len(query_reviews) == 0:
        print(f"Not found: '{query_album}'")
        return None

    query_artist_norm = df.loc[query_reviews.index[0], "artist_norm"]
    query_embedding = embeddings[query_reviews.index].mean(axis=0)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    sims = embeddings @ query_embedding
    sims[query_reviews.index] = -np.inf

    ranked = np.argsort(sims)[::-1]
    seen, results = set(), []
    for idx in ranked:
        artist_norm = df.loc[idx, "artist_norm"]
        if filter_same_artist and artist_norm == query_artist_norm:
            continue
        key = (artist_norm, df.loc[idx, "album_norm"])
        if key not in seen:
            seen.add(key)
            results.append({
                "album":   df.loc[idx, "album"],
                "artist":  df.loc[idx, "artist"],
                "source":  df.loc[idx, "dataset"],
                "score":   round(float(sims[idx]), 4),
                "snippet": df.loc[idx, "cleaned_text"][:300]
            })
        if len(results) >= k:
            break
    return results

recs = recommend("kid a", query_artist="radiohead")
for i, r in enumerate(recs, 1):
    print(f"\n{i}. {r['artist']} — {r['album']} ({r['source']}, score: {r['score']})")
    print(f"   {r['snippet'][:200]}...")

## Explainer prototype

Testing various forms of an explainer that describes the similarity between two unexpectedly connected albums using review themes/quotes as evidence.

In [ ]:
!pip install anthropic -q

import json, re
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say 'API working' and nothing else."}]
)
print(response.content[0].text)

In [ ]:
# Long form explainer (paragraph output)

def extract_dimensions(review_text):
    prompt = f"""Analyze this music review and extract the following. Return valid JSON only, no other text.

{{
  "sonic_descriptors": ["words or short phrases describing the sound or production style"],
  "themes": ["abstract concepts or ideas the album explores"],
  "critical_framing": "one short phrase describing how the critic positions this album",
  "formal_qualities": ["structural or compositional descriptions"],
  "genre_signals": ["specific genre or subgenre labels mentioned"]
}}

Review:
{review_text[:3000]}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.content[0].text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

# Test on Burial — Untrue
test_review = df[df["album_norm"] == "untrue"]["cleaned_text"].iloc[0]
print(json.dumps(extract_dimensions(test_review), indent=2))

In [ ]:
# Simplified explainer (bullet points output)

def explain_pair_simple(query_album, query_artist, rec_album, rec_artist, n=3, source=None):
    q_mask = (df["album_norm"] == query_album.lower().strip()) & (df["artist_norm"] == query_artist.lower().strip())
    r_mask = (df["album_norm"] == rec_album.lower().strip()) & (df["artist_norm"] == rec_artist.lower().strip())
    if source:
        q_mask &= df["dataset"] == source
        r_mask &= df["dataset"] == source

    q_reviews, r_reviews = df[q_mask], df[r_mask]
    if q_reviews.empty or r_reviews.empty:
        print("One or both albums not found.")
        return

    q_text, r_text   = q_reviews.iloc[0]["cleaned_text"], r_reviews.iloc[0]["cleaned_text"]
    q_source, r_source = q_reviews.iloc[0]["dataset"], r_reviews.iloc[0]["dataset"]
    q_label = f"{query_artist} — {query_album} ({q_source})"
    r_label = f"{rec_artist} — {rec_album} ({r_source})"

    prompt = f"""You are analyzing two music reviews to find their most meaningful shared qualities.

Identify up to {n} significant qualities that both albums genuinely share, based ONLY on what is actually written in the reviews.
If fewer than {n} genuine connections exist, list only those that are real.
If there are no meaningful connections, say so plainly.

For each shared quality:
- State it in one clear phrase
- Quote the specific language from each review that supports it (verbatim, in quotation marks, attributed to Review A or Review B)

Album A: {q_label}
Review A:
{q_text}

Album B: {r_label}
Review B:
{r_text}

Return a numbered list only. No introduction, no conclusion."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"\n{'='*60}\n  {q_label}\n  → {r_label}\n{'='*60}")
    print(f"\n{response.content[0].text.strip()}")

In [ ]:
# Test run simplified explainer

# Joni Mitchell — Blue → Bon Iver — For Emma, Forever Ago
mitchell_row = df[(df["artist_norm"] == "joni mitchell") & (df["album_norm"] == "blue")].iloc[0]
boniver_row  = df[(df["artist_norm"] == "bon iver") & (df["album_norm"] == "for emma, forever ago")].iloc[0]
explain_pair_simple(mitchell_row["album_norm"], mitchell_row["artist_norm"],
                    boniver_row["album_norm"],  boniver_row["artist_norm"])

# Radiohead — Kid A → Talk Talk — Spirit of Eden (Pitchfork only)
radiohead_row = df[(df["artist_norm"] == "radiohead") & (df["album_norm"] == "kid a") & (df["dataset"] == "pitchfork")].iloc[0]
talktalk_row  = df[(df["artist_norm"] == "talk talk") & (df["album_norm"] == "spirit of eden") & (df["dataset"] == "pitchfork")].iloc[0]
explain_pair_simple(radiohead_row["album_norm"], radiohead_row["artist_norm"],
                    talktalk_row["album_norm"],  talktalk_row["artist_norm"], source="pitchfork")

# Taylor Swift — Folklore → Sufjan Stevens — Carrie & Lowell (Pitchfork only)
folklore_row = df[(df["artist_norm"] == "taylor swift") & (df["album_norm"] == "folklore") & (df["dataset"] == "pitchfork")].iloc[0]
carrie_row   = df[(df["artist_norm"] == "sufjan stevens") & (df["album_norm"] == "carrie & lowell") & (df["dataset"] == "pitchfork")].iloc[0]
explain_pair_simple(folklore_row["album_norm"], folklore_row["artist_norm"],
                    carrie_row["album_norm"],   carrie_row["artist_name"], source="pitchfork")